# 03. 준지도학습(self-training) 검증

## 이 노트북이 답하려는 질문
라벨이 붙은 7,925행 중 불량은 71건(0.89%)뿐이다.
라벨이 없는 795,315행을 끌어다 쓰면 불량 판별이 나아지는가?

**self-training(자가학습)이란**
1. 라벨 있는 데이터로 모델을 만든다
2. 그 모델로 라벨 없는 데이터를 예측한다
3. 모델이 **확신하는** 예측만 골라 "의사 라벨(pseudo-label)"로 붙인다
4. 원래 라벨 + 의사 라벨로 다시 학습한다
5. 2~4를 반복한다

## 미리 말해두는 결론 (그래도 직접 확인할 것)
이 방법은 여기서 잘 안 될 가능성이 높다. 이유는 세 가지다.

1. **의사 라벨이 거의 전부 양품이 된다.** 불량이 1%도 안 되므로 모델이 확신하는 쪽은 항상 양품이다. 양품 수만 늘리면 불균형이 더 심해진다.
2. **기준 모델이 약하다.** 쇼트 단위로 제대로 나눠 평가하면 CN7 F1 0.50, RG3 0.17이다. 이 모델이 붙인 라벨은 틀린 것이 섞여 있고, 그 오류가 다음 학습에 그대로 들어간다(확증편향).
3. **불량 세 종류 중 두 종류는 공정변수에 신호가 없다.** 가스는 금형 벤트, 초기허용불량은 기동 시각이 원인이다. 라벨 없는 데이터를 아무리 넣어도 없는 신호가 생기지는 않는다.

**그래도 돌려보는 이유**: 보고서의 "3. 영향요인 및 오류분석"과 "5. 창의성·차별성" 항목에서
"시도했으나 왜 안 되는지 수치로 보였다"는 것이 "안 해봤다"보다 훨씬 강하다.
안 되는 이유를 숫자로 남기는 것이 이 노트북의 산출물이다.

## 반드시 지킬 것
- **평가는 언제나 원래(진짜) 라벨로만** 한다. 의사 라벨로 평가하면 자기 답안지를 자기가 채점하는 꼴이다.
- **분할은 쇼트 단위로** 한다. 같은 쇼트의 LH/RH는 공정변수 24개가 완전히 같다. 하나가 train, 하나가 test로 가면 정답을 베끼는 것이다.
- **오버샘플링은 train fold 안에서만** 한다. 나누기 전에 하면 가이드북과 같은 누설이 생긴다.

## 0. 준비

In [ ]:
import sys, warnings
sys.path.append("..")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

from src.load import load_labeled, load_moldset, PROC_COLS, RAW

RNG = 42
np.random.seed(RNG)

# 어떤 파일이 있는지 먼저 확인
for p in sorted(RAW.glob("*.csv")):
    print(f"{p.name:35s} {p.stat().st_size/1e6:8.1f} MB")

## 1. 데이터 준비

2차 가공본(`moldset_*`)이 제출 모델의 입력 형식이다. 그런데 여기엔 시각·LH/RH가 없어서
**쇼트 단위로 나눌 수가 없다.** 그래서 1차 가공본에서 쇼트 번호를 가져와 붙인다.

붙이는 방법: 공정변수 24개 값의 조합이 같으면 같은 쇼트로 본다.
(같은 쇼트의 LH/RH는 24개 값이 완전히 동일하므로 이 방법이 성립한다)

In [ ]:
PRODUCT = "CN7"     # "RG3" 로 바꿔 한 번 더 돌린다

lab1 = load_labeled(PRODUCT)                  # 1차 가공본 (시각·LH/RH·불량유형 있음)
print(f"1차 가공본 {PRODUCT}: {len(lab1)}행, 불량 {lab1['y'].sum()}건, 쇼트 {lab1['shot'].nunique()}개")

lab2 = load_moldset(PRODUCT, labeled=True)    # 2차 가공본 (제출 형식)
print(f"2차 가공본 {PRODUCT}: {len(lab2)}행")
print(f"컬럼: {list(lab2.columns)}")

In [ ]:
def make_key(df, cols):
    """공정변수 값 조합을 문자열 키로. 부동소수 오차를 없애려고 반올림 후 해시."""
    sub = df[cols].round(4)
    return pd.util.hash_pandas_object(sub, index=False).astype(str)

# 두 파일에 공통으로 있는 변수만 사용
common = [c for c in PROC_COLS if c in lab1.columns and c in lab2.columns]
print(f"공통 변수 {len(common)}개")

lab1 = lab1.copy()
lab1["key"] = make_key(lab1, common)

# key -> shot 사전
key2shot = lab1.drop_duplicates("key").set_index("key")["shot"].to_dict()

lab2 = lab2.copy()
lab2["key"] = make_key(lab2, common)
lab2["shot"] = lab2["key"].map(key2shot)

matched = lab2["shot"].notna().mean()
print(f"쇼트 번호가 붙은 비율: {matched:.1%}")
if matched < 0.9:
    print("!! 매칭률이 낮다. 2차 가공본이 표준화되어 값이 달라졌을 수 있다.")
    print("   그럴 땐 아래 셀의 FALLBACK 을 쓴다.")

In [ ]:
# FALLBACK: 매칭이 안 되면 1차 가공본으로 직접 학습한다.
# 값이 표준화되어도 나무 기반 모델(RandomForest)은 단조변환에 영향받지 않으므로
# 결론(어느 쪽이 더 나은가)은 그대로 유지된다.
USE_FALLBACK = matched < 0.9

if USE_FALLBACK:
    X = lab1[common].values
    y = lab1["y"].values
    groups = lab1["shot"].values
    print("FALLBACK: 1차 가공본 사용")
else:
    ok = lab2["shot"].notna()
    lab2 = lab2[ok]
    ycol = "PassOrFail" if "PassOrFail" in lab2.columns else "y"
    X = lab2[common].values
    y = (lab2[ycol] == 1).astype(int).values if lab2[ycol].dtype != object \
        else (lab2[ycol] == "N").astype(int).values
    groups = lab2["shot"].values.astype(int)
    print("2차 가공본 사용")

print(f"X {X.shape}, 불량 {y.sum()}건 ({y.mean():.2%}), 쇼트 {len(np.unique(groups))}개")

## 2. 공용 함수

### 왜 F1 임계값을 따로 찾는가
불량이 0.89%면 기본 임계값 0.5에서는 아무것도 불량으로 안 나온다.
확률을 뽑아 임계값을 훑으면서 F1이 가장 높은 지점을 쓴다.
**단, 임계값은 train fold 안에서 정하고 test fold에 적용해야 한다.**
여기서는 비교가 목적이라 모든 방법에 동일한 규칙을 적용해 상대비교만 한다.

In [ ]:
def oversample(Xtr, ytr, ratio=1.0, seed=RNG):
    """소수 클래스를 복제해 늘린다. train fold 안에서만 호출할 것."""
    rs = np.random.RandomState(seed)
    pos = np.where(ytr == 1)[0]
    neg = np.where(ytr == 0)[0]
    if len(pos) == 0:
        return Xtr, ytr
    n_want = int(len(neg) * ratio)
    pick = rs.choice(pos, size=n_want, replace=True)
    idx = np.concatenate([neg, pick])
    rs.shuffle(idx)
    return Xtr[idx], ytr[idx]


def best_f1(ytrue, prob):
    """임계값을 훑어 최고 F1과 그때의 임계값을 돌려준다."""
    best = (0.0, 0.5)
    for t in np.unique(np.round(prob, 3)):
        f = f1_score(ytrue, (prob >= t).astype(int), zero_division=0)
        if f > best[0]:
            best = (f, t)
    return best


def make_model(seed=RNG):
    return RandomForestClassifier(
        n_estimators=300, max_depth=None, min_samples_leaf=2,
        class_weight="balanced_subsample", n_jobs=-1, random_state=seed)


def evaluate(X, y, groups, X_extra=None, y_extra=None, n_splits=5, tag=""):
    """쇼트 단위 교차검증. X_extra/y_extra(의사 라벨)는 train fold 에만 더한다."""
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RNG)
    rows = []
    for k, (tr, te) in enumerate(cv.split(X, y, groups)):
        Xtr, ytr = X[tr], y[tr]
        if X_extra is not None and len(X_extra):
            Xtr = np.vstack([Xtr, X_extra])
            ytr = np.concatenate([ytr, y_extra])
        Xtr, ytr = oversample(Xtr, ytr)
        m = make_model(seed=RNG + k).fit(Xtr, ytr)
        prob = m.predict_proba(X[te])[:, 1]
        f, t = best_f1(y[te], prob)
        pred = (prob >= t).astype(int)
        rows.append(dict(
            fold=k, f1=f, thr=t,
            precision=precision_score(y[te], pred, zero_division=0),
            recall=recall_score(y[te], pred, zero_division=0),
            n_test_pos=int(y[te].sum())))
    r = pd.DataFrame(rows)
    print(f"[{tag}] F1 {r.f1.mean():.3f} ± {r.f1.std():.3f} | "
          f"P {r.precision.mean():.3f} | R {r.recall.mean():.3f} | "
          f"fold당 test 불량 {r.n_test_pos.mean():.1f}건")
    return r

## 3. 기준선 — 라벨만으로 학습

비교의 출발점. 여기 숫자보다 나아지지 않으면 준지도학습은 의미가 없다.

In [ ]:
res_base = evaluate(X, y, groups, tag="기준선(라벨만)")
res_base

## 4. 가이드북 방식과의 대조 — 왜 F1 1.00이 나왔는가

가이드북은 **나누기 전에** 오버샘플링을 했다. 그러면 같은 불량 행의 복제본이
train과 test 양쪽에 들어간다. test에서 맞히는 게 아니라 외운 것을 다시 보는 것이다.

이 셀은 그 방식을 그대로 재현해 숫자를 대조한다. **제출 모델에는 절대 쓰지 않는다.**

In [ ]:
from sklearn.model_selection import train_test_split

# 가이드북 방식: 먼저 오버샘플 → 그 다음 분할
Xo, yo = oversample(X, y)
Xtr, Xte, ytr, yte = train_test_split(Xo, yo, test_size=0.2,
                                      stratify=yo, random_state=RNG)
m = make_model().fit(Xtr, ytr)
pred = m.predict(Xte)
f_leak = f1_score(yte, pred, zero_division=0)

print(f"가이드북 방식 F1 : {f_leak:.3f}   (test 불량 {int(yte.sum())}건)")
print(f"올바른 방식  F1 : {res_base.f1.mean():.3f}   (실제 존재하는 불량 {int(y.sum())}건)")
print()
print(f"test 에 있다고 주장하는 불량 {int(yte.sum())}건 vs 데이터 전체 불량 {int(y.sum())}건")
print("→ 없는 불량을 만들어 채점한 것이다.")

## 5. self-training

### 어떻게 확신을 재는가
RandomForest 의 예측 확률 `p` 에서 **엔트로피**를 쓴다.

$$H = -p\log p - (1-p)\log(1-p)$$

p가 0이나 1에 가까울수록 H는 0에 가깝다 = 확신한다.
H가 작은 순으로 정렬해 상위 일부만 의사 라벨로 채택한다.

### 반복마다 얼마나 넣는가
남은 라벨 없는 데이터의 10%씩, 5회 반복.
한 번에 다 넣으면 틀린 라벨이 한꺼번에 들어와 되돌릴 수 없다.

In [ ]:
# 라벨 없는 데이터 읽기. 795,315행 전부는 무겁다. 표본으로 시작한다.
N_UNLAB = 50_000

try:
    unlab = load_moldset(PRODUCT, labeled=False)
except FileNotFoundError:
    print("moldset_unlabeled_* 이 없다. 1차 unlabeled_data.csv 에서 읽는다.")
    unlab = pd.read_csv(RAW / "unlabeled_data.csv", nrows=400_000)
    unlab = unlab[unlab["PART_NAME"].str.startswith(PRODUCT, na=False)]

unlab = unlab[[c for c in common if c in unlab.columns]].dropna()
print(f"라벨 없는 데이터 {len(unlab):,}행")

if len(unlab) > N_UNLAB:
    unlab = unlab.sample(N_UNLAB, random_state=RNG)
U = unlab[common].values
print(f"사용할 표본 {U.shape}")

In [ ]:
def entropy(p):
    p = np.clip(p, 1e-9, 1 - 1e-9)
    return -(p * np.log(p) + (1 - p) * np.log(1 - p))


def self_train(X, y, groups, U, n_iter=5, frac=0.10, thr_conf=0.95):
    """self-training 을 n_iter 회 돌리며 매 회 성능을 기록한다.

    frac      : 매 반복마다 남은 U 중 몇 %를 채택할지
    thr_conf  : 이 확률 이상/이하여야 의사 라벨을 붙인다
    """
    U_left = U.copy()
    PX = np.empty((0, X.shape[1]))
    Py = np.empty(0, dtype=int)
    log = []

    for it in range(1, n_iter + 1):
        # 현재까지의 의사 라벨을 더해 모델을 만든다
        Xtr = np.vstack([X, PX]) if len(PX) else X
        ytr = np.concatenate([y, Py]) if len(Py) else y
        Xtr_o, ytr_o = oversample(Xtr, ytr)
        m = make_model(seed=RNG + it).fit(Xtr_o, ytr_o)

        if len(U_left) == 0:
            break

        p = m.predict_proba(U_left)[:, 1]
        H = entropy(p)
        n_take = max(1, int(len(U_left) * frac))
        order = np.argsort(H)[:n_take]           # 확신하는 순
        take = order[(p[order] >= thr_conf) | (p[order] <= 1 - thr_conf)]

        if len(take) == 0:
            print(f"반복 {it}: 임계 {thr_conf} 를 넘는 확신이 없다. 중단.")
            break

        new_y = (p[take] >= 0.5).astype(int)
        PX = np.vstack([PX, U_left[take]])
        Py = np.concatenate([Py, new_y])
        U_left = np.delete(U_left, take, axis=0)

        # 평가는 언제나 진짜 라벨로
        r = evaluate(X, y, groups, X_extra=PX, y_extra=Py,
                     tag=f"self-training {it}회")
        log.append(dict(iter=it,
                        f1=r.f1.mean(), precision=r.precision.mean(),
                        recall=r.recall.mean(),
                        pseudo_total=len(Py),
                        pseudo_defect=int(Py.sum()),
                        pseudo_defect_ratio=Py.mean()))
        print(f"    의사 라벨 누적 {len(Py):,}개 중 불량 {int(Py.sum())}개 "
              f"({Py.mean():.4%})\n")

    return pd.DataFrame(log), PX, Py

In [ ]:
log, PX, Py = self_train(X, y, groups, U, n_iter=5, frac=0.10, thr_conf=0.95)
log

## 6. 해석 — 왜 안 되는가(혹은 되는가)

아래 세 숫자를 확인한다.

1. **`pseudo_defect`** — 의사 라벨 중 불량으로 붙은 개수.
   0이거나 한 자릿수라면, 795,315행을 다 넣어도 모델이 배울 불량 사례가 안 늘어난다.
   준지도학습이 여기서 통하지 않는 **직접적인 증거**다.

2. **`f1` 의 추이** — 반복할수록 떨어지면 확증편향이다.
   기준 모델이 틀린 것을 확신하고, 그 틀린 라벨로 다시 배워 더 틀린다.

3. **`recall`** — 양품 의사 라벨만 쌓이면 불균형이 심해져 재현율이 먼저 떨어진다.

In [ ]:
import matplotlib.pyplot as plt
from src.plotting import setup, save
setup()

if len(log):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    ax = axes[0]
    ax.axhline(res_base.f1.mean(), color="gray", ls="--", label="기준선(라벨만)")
    ax.plot(log["iter"], log["f1"], "o-", label="self-training")
    ax.plot(log["iter"], log["recall"], "s-", alpha=.6, label="재현율")
    ax.set_xlabel("반복 횟수"); ax.set_ylabel("점수")
    ax.set_title(f"{PRODUCT} — 반복에 따른 성능")
    ax.legend(); ax.grid(alpha=.3)

    ax = axes[1]
    ax.bar(log["iter"], log["pseudo_total"], label="의사 라벨 전체", color="#bbb")
    ax.bar(log["iter"], log["pseudo_defect"], label="그중 불량", color="crimson")
    ax.set_xlabel("반복 횟수"); ax.set_ylabel("개수")
    ax.set_title("의사 라벨 구성")
    ax.legend(); ax.grid(alpha=.3)

    plt.tight_layout()
    save(fig, f"selftrain_{PRODUCT}")
    plt.show()

    print(f"\n최종 의사 라벨 {len(Py):,}개 중 불량 {int(Py.sum())}개")
    print(f"원래 라벨의 불량 {int(y.sum())}개 대비 증가분: {int(Py.sum())}개")
else:
    print("반복이 한 번도 성립하지 않았다. thr_conf 를 낮춰 다시 시도하거나,")
    print("그 자체가 '모델이 라벨 없는 데이터에 대해 확신하지 못한다'는 결과다.")

## 7. 대안 — 약지도학습(weak supervision)

self-training 이 실패하는 이유는 "라벨이 없다"가 아니라 **"불량 사례가 없다"**이다.
라벨 없는 데이터를 아무리 넣어도 불량 사례 수는 안 늘어난다.

그런데 1차 가공본에는 `ERR_FACT_QTY` (그날의 불량 수량)가 있다.
이건 행 단위 라벨은 아니지만 **"이 날 이 제품에서 불량이 N건 나왔다"**는 약한 라벨이다.

아래 셀은 그 수량이 실제 라벨된 불량 수와 맞는지 확인한다.
**맞으면** 라벨 없는 구간에도 "이 날 N건 있었다"를 적용해 학습할 수 있다.
**안 맞으면** 이 경로도 막힌 것이고, 그 사실도 보고서에 쓸 거리다.

In [ ]:
full = load_labeled(PRODUCT)
if "ERR_FACT_QTY" in full.columns:
    chk = (full.groupby("date")
               .agg(라벨불량=("y", "sum"),
                    ERR_FACT_QTY최대=("ERR_FACT_QTY", "max"),
                    행수=("y", "size"))
               .reset_index())
    chk["차이"] = chk["ERR_FACT_QTY최대"] - chk["라벨불량"]
    print(chk.to_string(index=False))
    print()
    agree = (chk["차이"] == 0).mean()
    print(f"일치하는 날 비율: {agree:.1%}")
    if agree > 0.7:
        print("→ 약지도학습 경로가 열려 있다. 10/1 마일스톤에서 검증한다.")
    else:
        print("→ 수량이 라벨과 맞지 않는다. 다른 기준(예: 공정 전체 생산량)일 수 있다.")
else:
    print("ERR_FACT_QTY 컬럼이 없다.")

## 8. 회의용 정리 (숫자 채워 넣기)

| 항목 | 값 |
|---|---|
| 기준선 F1 (라벨만, 쇼트 단위 CV) | |
| 가이드북 방식 F1 (누설) | |
| self-training 5회 후 F1 | |
| 의사 라벨 총 개수 | |
| 그중 불량으로 붙은 개수 | |
| ERR_FACT_QTY 일치 날짜 비율 | |

### 회의에서 할 말 세 줄
1. 가이드북의 F1 1.00은 나누기 전에 오버샘플링해서 나온 숫자다. 제대로 나누면 CN7 0.50, RG3 0.17이다.
2. self-training 을 돌려봤는데 의사 라벨 ○○개 중 불량이 ○개였다. 불량 사례가 안 늘어나니 성능도 안 오른다.
3. 쓸 만한 방향은 (a) 불량 세 종류를 나눠서 각각 다른 방법으로 잡기, (b) ERR_FACT_QTY 로 약지도학습이다.

### 다음 할 일
- [ ] `PRODUCT = "RG3"` 로 바꿔 한 번 더 돌린다
- [ ] 결과 숫자를 `docs/findings.md` 에 기록한다
- [ ] `N_UNLAB` 를 늘려도(예: 200,000) 결론이 같은지 확인한다